# Vapnik–Chervonenkis (VC) Dimension - Starter Notebook
Demonstrates the VC dimension concept: the largest set of points a hypothesis class can shatter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification
from itertools import product

In [ ]:
# VC dimension of linear classifiers in R^2 = 3
# Demonstrate shattering 3 points in R^2

# 3 points in R^2 that can be shattered by a linear classifier
points_3 = np.array([[0, 0], [1, 0], [0.5, 1]])

labels_all = list(product([0, 1], repeat=3))  # all 2^3=8 labelings
print(f'Testing all {len(labels_all)} labelings of 3 points:')

shattering_ok = 0
for labels in labels_all:
    y = np.array(labels)
    # Skip all-same (trivial for linear)
    clf = SVC(kernel='linear')
    # Use leave-one-out: just try to fit
    try:
        clf.fit(points_3, y)
        pred = clf.predict(points_3)
        if np.all(pred == y):
            shattering_ok += 1
    except Exception:
        pass

print(f'Labelings correctly separated: {shattering_ok}/{len(labels_all)}')
print('VC dimension of linear classifier in R^2 = 3 (can shatter 3 points)')

In [ ]:
# Visualize shattering of 3 points
fig, axes = plt.subplots(2, 4, figsize=(12, 5))
axes = axes.flatten()

for ax, labels in zip(axes, labels_all):
    y = np.array(labels)
    colors = ['red' if yi == 1 else 'blue' for yi in y]
    ax.scatter(points_3[:, 0], points_3[:, 1], c=colors, s=100, edgecolors='k', zorder=3)
    ax.set_title(str(labels), fontsize=8)
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.set_xticks([])
    ax.set_yticks([])
    # Draw separating line if not all-same
    if len(set(labels)) == 2:
        clf = SVC(kernel='linear')
        clf.fit(points_3, y)
        xx = np.linspace(-0.3, 1.3, 100)
        w = clf.coef_[0]
        b = clf.intercept_[0]
        if abs(w[1]) > 1e-6:
            yy = -(w[0] * xx + b) / w[1]
            ax.plot(xx, yy, 'g-', linewidth=1)

red_p = mpatches.Patch(color='red', label='Class 1')
blue_p = mpatches.Patch(color='blue', label='Class 0')
fig.suptitle('Shattering 3 points: all 8 labelings separated by a line', fontsize=11)
fig.legend(handles=[red_p, blue_p], loc='lower center', ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# VC dimension vs model complexity: generalization bound
# Sauer's lemma: growth function m_H(N) <= (N*e/d)^d for N >= d
# Generalization bound ~= sqrt((d(ln(2N/d)+1) - ln(delta)) / N)

N = np.arange(10, 500)
delta = 0.05
vc_dims = [1, 3, 5, 10, 20]

plt.figure(figsize=(8, 5))
for d in vc_dims:
    bound = np.sqrt((d * (np.log(2*N/d) + 1) + np.log(1/delta)) / N)
    plt.plot(N, bound, label=f'VC dim={d}')

plt.xlabel('Training set size N')
plt.ylabel('Generalization error bound')
plt.title('VC Bound: Higher VC dim → more data needed for generalization')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Empirical: test train/test gap vs decision tree depth (proxy for VC dim)
np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2, n_informative=2,
                           n_redundant=0, random_state=42)
depths = range(1, 16)
train_scores, test_scores = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    cv = cross_val_score(clf, X, y, cv=5)
    clf.fit(X, y)
    train_scores.append(clf.score(X, y))
    test_scores.append(cv.mean())

plt.figure(figsize=(7, 4))
plt.plot(depths, train_scores, 'o-', label='Train accuracy')
plt.plot(depths, test_scores, 's--', label='CV accuracy')
plt.xlabel('Max depth (proxy VC complexity)')
plt.ylabel('Accuracy')
plt.title('Bias-Variance via VC dimension: Decision Tree depth')
plt.legend()
plt.tight_layout()
plt.show()